# Outbreak in Numeria: The Null Fever Simulation

The city of **Numeria** is isolated for the next **180 days**. The mountain passes are closed, and the council must choose policies before outside medical caravans can arrive. A new disease, **Null Fever**, is spreading through the city.

Run the **setup cells** first. After that, each mission can be run separately.

## Setup 1 — Libraries and plotting settings

In [ ]:
import math
from dataclasses import dataclass, replace
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display, HTML


plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "font.size": 11,
})

np.set_printoptions(precision=4, suppress=True)

## Setup 2 — Model, numerical methods, and helper functions

The epidemic state is

$$
Y(t)=(S(t),V(t),E(t),I(t),Q(t),R(t),C(t)).
$$

The variable $C(t)$ is cumulative infections.

In [ ]:
COMPARTMENTS = ("S", "V", "E", "I", "Q", "R", "C")
IDX = {name: i for i, name in enumerate(COMPARTMENTS)}
POPULATION_IDXS = [IDX[name] for name in ("S", "V", "E", "I", "Q", "R")]


@dataclass(frozen=True)
class Params:
    N: float = 100_000.0
    beta0: float = 0.75
    a: float = 0.10
    kappa: float = 25.0
    rho: float = 0.05
    eta: float = 0.80
    nu: float = 0.0005
    omega: float = 1.0 / 240.0
    sigma: float = 1.0 / 4.0
    gamma: float = 1.0 / 8.0
    delta: float = 0.08
    gamma_q: float = 1.0 / 10.0


def initial_state():
    return np.array([99_940.0, 0.0, 40.0, 20.0, 0.0, 0.0, 60.0], dtype=float)


def beta_eff(t, y, p):
    I = max(float(y[IDX["I"]]), 0.0)
    weekly_factor = 1.0 + p.a * math.sin(2.0 * math.pi * t / 7.0)
    behavior_factor = 1.0 + p.kappa * I / p.N
    return p.beta0 * weekly_factor / behavior_factor


def force_of_infection(t, y, p):
    I = float(y[IDX["I"]])
    Q = float(y[IDX["Q"]])
    return beta_eff(t, y, p) * (I + p.rho * Q) / p.N


def rhs(t, y, p):
    S, V, E, I, Q, R, C = y
    lam = force_of_infection(t, y, p)

    infections_from_S = lam * S
    infections_from_V = (1.0 - p.eta) * lam * V
    incidence = infections_from_S + infections_from_V

    dS = -infections_from_S - p.nu * S + p.omega * R
    dV = p.nu * S - infections_from_V
    dE = incidence - p.sigma * E
    dI = p.sigma * E - p.gamma * I - p.delta * I
    dQ = p.delta * I - p.gamma_q * Q
    dR = p.gamma * I + p.gamma_q * Q - p.omega * R
    dC = incidence

    return np.array([dS, dV, dE, dI, dQ, dR, dC], dtype=float)


def conservation_defect(t, y, p):
    return float(np.sum(rhs(t, y, p)[POPULATION_IDXS]))


def euler_step(t, y, dt, p):
    return y + dt * rhs(t, y, p)


def heun_step(t, y, dt, p):
    k1 = rhs(t, y, p)
    k2 = rhs(t + dt, y + dt * k1, p)
    return y + 0.5 * dt * (k1 + k2)


def rk4_step(t, y, dt, p):
    k1 = rhs(t, y, p)
    k2 = rhs(t + 0.5 * dt, y + 0.5 * dt * k1, p)
    k3 = rhs(t + 0.5 * dt, y + 0.5 * dt * k2, p)
    k4 = rhs(t + dt, y + dt * k3, p)
    return y + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)



#Numericals schemes
METHODS = {
    "Euler": euler_step,
    "Heun": heun_step,
    "RK4": rk4_step,
}


def simulate(p=None, method="RK4", dt=0.25, T=180.0, y0=None):
    """Simulate from t=0 to T.
    """
    if p is None:
        p = Params()
    if y0 is None:
        y0 = initial_state()
    step = METHODS[method]

    nsteps = int(math.ceil(T / dt))
    t = np.empty(nsteps + 1)
    Y = np.empty((nsteps + 1, len(COMPARTMENTS)))
    t[0] = 0.0
    Y[0] = np.array(y0, dtype=float)

    current_t = 0.0
    current_y = Y[0].copy()
    for n in range(nsteps):
        h = min(float(dt), T - current_t)
        current_y = step(current_t, current_y, h, p)
        current_t = current_t + h
        t[n + 1] = current_t
        Y[n + 1] = current_y

    return t, Y


def hospital_load(Y):
    return 0.04 * Y[:, IDX["I"]] + 0.10 * Y[:, IDX["Q"]]


def population(Y):
    return np.sum(Y[:, POPULATION_IDXS], axis=1)


def summarize(t, Y, p=None, hospital_capacity=600.0):
    if p is None:
        p = Params()
    I = Y[:, IDX["I"]]
    H = hospital_load(Y)
    pop_error = np.max(np.abs(population(Y) - p.N))

    if len(t) > 1:
        overload_days = np.sum((H[:-1] > hospital_capacity) * np.diff(t))
    else:
        overload_days = 0.0

    return {
        "peak_I": float(np.max(I)),
        "day_peak_I": float(t[np.argmax(I)]),
        "peak_hospital_load": float(np.max(H)),
        "day_peak_hospital_load": float(t[np.argmax(H)]),
        "overload_days": float(overload_days),
        "final_cumulative_cases": float(Y[-1, IDX["C"]]),
        "minimum_compartment": float(np.min(Y[:, POPULATION_IDXS])),
        "max_population_error": float(pop_error),
    }


def interp_component(t_source, Y_source, t_target, name):
    return np.interp(t_target, t_source, Y_source[:, IDX[name]])


def error_against_reference(t, Y, t_ref, Y_ref, name="I"):
    ref = interp_component(t_ref, Y_ref, t, name)
    return float(np.max(np.abs(Y[:, IDX[name]] - ref)))


def plot_compartments(t, Y, title="Compartment evolution"):
    fig, ax = plt.subplots(figsize=(9, 5))
    for name in ["S", "V", "E", "I", "Q", "R"]:
        ax.plot(t, Y[:, IDX[name]], label=name)
    ax.set_title(title)
    ax.set_xlabel("Time in days")
    ax.set_ylabel("People")
    ax.legend(ncol=3)
    plt.show()


def plot_dashboard(t, Y, title="City dashboard", hospital_capacity=600.0):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t, Y[:, IDX["I"]], label="Infectious I(t)")
    ax.plot(t, Y[:, IDX["Q"]], label="Isolated Q(t)")
    ax.plot(t, hospital_load(Y), label="Hospital load H(t)")
    ax.axhline(hospital_capacity, linestyle="--", label="Hospital capacity")
    ax.set_title(title)
    ax.set_xlabel("Time in days")
    ax.set_ylabel("People / beds")
    ax.legend()
    plt.show()


def display_summary(title, t, Y, p=None):
    print(title)
    display(pd.DataFrame([summarize(t, Y, p)]).T.rename(columns={0: "value"}))

# Mission 1 — Decode the Null Fever model

Before the city trusts a simulator, it must understand what is being simulated.

In [ ]:
p = Params()
y0 = initial_state()
dy0 = rhs(0.0, y0, p)

pd.DataFrame({
    "compartment": COMPARTMENTS,
    "initial value": y0,
    "initial derivative": dy0,
})

In [ ]:
print("Conservation defect at t=0:", conservation_defect(0.0, y0, p))
print("Initial population:", np.sum(y0[POPULATION_IDXS]))
print("Initial cumulative infections C(0):", y0[IDX["C"]])

# Mission 2 — Build the first 180-day forecast

The council asks for a forecast over the 180 days during which Numeria is isolated.

Use explicit Euler with $\Delta t=0.25$ days. Then compare it with a small-step RK4 reference.

### Questions

1. On which day does $I(t)$ reach its maximum?
2. What is the maximum value of $I(t)$?
3. Does the hospital load exceed capacity?

In [ ]:
t_eu, Y_eu = simulate(p, method="Euler", dt=0.25, T=180.0)
display_summary("Euler baseline, dt=0.25", t_eu, Y_eu, p)
plot_compartments(t_eu, Y_eu, "Null Fever forecast: Euler, dt=0.25")
plot_dashboard(t_eu, Y_eu, "Hospital dashboard: Euler, dt=0.25")

In [ ]:
t_ref, Y_ref = simulate(p, method="RK4", dt=0.02, T=180.0)
display_summary("RK4 reference, dt=0.02", t_ref, Y_ref, p)
plot_dashboard(t_ref, Y_ref, "Hospital dashboard: RK4 reference")

# Mission 3 — The simulation can lie

The same model is simulated with different Euler time steps. A larger step is faster, but it may create problems ...

### Question

When does the simulation first become suspicious?



In [ ]:
dts = [0.25, 0.5, 1.0, 2.0, 4.0, 6.0, 8.0]
rows = []
trajectories = {}

for dt in dts:
    t, Y = simulate(p, method="Euler", dt=dt, T=180.0)
    rows.append({"dt": dt, **summarize(t, Y, p)})
    trajectories[dt] = (t, Y)

failure_table = pd.DataFrame(rows)
display(failure_table)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(failure_table["dt"], failure_table["peak_I"], marker="o")
ax.set_xscale("log", base=2)
ax.set_title("Euler time step versus peak infectious count")
ax.set_xlabel("dt in days")
ax.set_ylabel("Peak I(t)")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(failure_table["dt"], failure_table["minimum_compartment"], marker="o")
ax.axhline(0, linestyle="--")
ax.set_xscale("log", base=2)
ax.set_title("Euler time step versus smallest compartment value")
ax.set_xlabel("dt in days")
ax.set_ylabel("Minimum of S,V,E,I,Q,R")
plt.show()

In [ ]:
dt_bad = 8.0
t_bad, Y_bad = trajectories[dt_bad]
plot_dashboard(t_bad, Y_bad, f"Bad forecast: Euler, dt={dt_bad}")

# Mission 4 — The quarantine shock

Now the city changes only three rates:

$$
\delta=2.5,\qquad \gamma_Q=1.0,\qquad \sigma=1.5.
$$

Exposed people become infectious quickly, infectious people are isolated quickly, and isolated people recover quickly. The model now contains faster time scales.

### Questions

1. Which values of $\Delta t$ still work for Euler?
2. Which values create oscillations or negative compartments?
<!-- 3. What does this experiment reveal about stiffness? -->

In [ ]:
p_stiff = replace(p, delta=2.5, gamma_q=1.0, sigma=1.5)
stiff_dts = [0.05, 0.1, 0.25, 0.5, 0.6, 0.7, 1.0]
rows = []

for dt in stiff_dts:
    t, Y = simulate(p_stiff, method="Euler", dt=dt, T=180.0)
    rows.append({"dt": dt, **summarize(t, Y, p_stiff)})

stiff_table = pd.DataFrame(rows)
display(stiff_table)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(stiff_table["dt"], stiff_table["minimum_compartment"], marker="o")
ax.axhline(0, linestyle="--")
ax.set_title("Stiffness test: Euler minimum compartment")
ax.set_xlabel("dt in days")
ax.set_ylabel("Minimum of S,V,E,I,Q,R")
plt.show()

# Mission 5 — Three engines for the same city: Euler, Heun, RK4

Euler is simple. Heun is a second-order predictor-corrector method. RK4 is a fourth-order method.

We compare all three against a small-step RK4 reference.

### Questions

1. Which method is most accurate for the same time step?
2. Does a higher-order method completely remove stability problems?
3. Why might your team still use simple methods in large simulations?

In [ ]:
# Reference solution.
t_ref, Y_ref = simulate(p, method="RK4", dt=0.01, T=180.0)

compare_dts = [0.25, 0.5, 1.0, 2.0]
rows = []
for method in ["Euler", "Heun", "RK4"]:
    for dt in compare_dts:
        t, Y = simulate(p, method=method, dt=dt, T=180.0)
        rows.append({
            "method": method,
            "dt": dt,
            "error_in_I": error_against_reference(t, Y, t_ref, Y_ref, "I"),
            "minimum_compartment": summarize(t, Y, p)["minimum_compartment"],
        })

method_table = pd.DataFrame(rows)
display(method_table)

fig, ax = plt.subplots(figsize=(8, 4.5))
for method in ["Euler", "Heun", "RK4"]:
    data = method_table[method_table["method"] == method]
    ax.loglog(data["dt"], data["error_in_I"], marker="o", label=method)
ax.set_title("Method comparison: error in infectious curve")
ax.set_xlabel("dt in days")
ax.set_ylabel("max |I_method - I_reference|")
ax.legend()
plt.show()

# Mission 6 — Measure convergence order

A numerical method has order $p$ if the error behaves approximately like

$$
\text{error}(\Delta t)\approx C(\Delta t)^p.
$$

On a log-log plot, this looks like a straight line of slope $p$.

If two time steps $\Delta t_1$ and $\Delta t_2$ give errors $e_1$ and $e_2$, the the order can be estimated by

$$
p\approx \frac{\log(e_1/e_2)}{\log(\Delta t_1/\Delta t_2)}.
$$

### Question

What observed order do you get for each scheme?
<!-- 2. What observed order do you get for Heun?
3. What observed order do you get for RK4? -->
<!-- 4. Why might the observed order be imperfect for large time steps? -->

In [ ]:
t_ref, Y_ref = simulate(p, method="RK4", dt=0.01, T=180.0)
conv_dts = [1.6, 0.8, 0.4, 0.2, 0.1]
rows = []

for method in ["Euler", "Heun", "RK4"]:
    for dt in conv_dts:
        t, Y = simulate(p, method=method, dt=dt, T=180.0)
        rows.append({
            "method": method,
            "dt": dt,
            "error_in_I": error_against_reference(t, Y, t_ref, Y_ref, "I"),
        })

conv_table = pd.DataFrame(rows)
display(conv_table)

fig, ax = plt.subplots(figsize=(8, 4.5))
for method in ["Euler", "Heun", "RK4"]:
    data = conv_table[conv_table["method"] == method]
    ax.loglog(data["dt"], data["error_in_I"], marker="o", label=method)
ax.set_title("Convergence plot")
ax.set_xlabel("dt in days")
ax.set_ylabel("max |I_method - I_reference|")
ax.legend()
plt.show()

orders = []
for method in ["Euler", "Heun", "RK4"]:
    data = conv_table[conv_table["method"] == method].sort_values("dt", ascending=False)
    dts_arr = data["dt"].to_numpy()
    errs = data["error_in_I"].to_numpy()
    for j in range(len(dts_arr) - 1):
        observed_order = math.log(errs[j] / errs[j + 1]) / math.log(dts_arr[j] / dts_arr[j + 1])
        orders.append({
            "method": method,
            "from_dt": dts_arr[j],
            "to_dt": dts_arr[j + 1],
            "observed_order": observed_order,
        })

order_table = pd.DataFrame(orders)
display(order_table)

# Mission 7 — Intervention cards

The city has limited resources. Each policy card has a cost. The council can spend at most **5 points**.

The goal is not only to reduce infections, but also to avoid hospital overload.

### Questions

1. Which policy combination gives the smallest hospital overload?
2. Which gives the smallest cumulative number of infections?
3. Is the best policy obvious, or is there a tradeoff?
4. Can several weak interventions combine into a strong one?

In [ ]:
def policy_cards():
    return [
        {
            "name": "vaccine sprint",
            "cost": 2,
            "apply": lambda q: replace(q, nu=q.nu + 0.0007),
        },
        {
            "name": "booster campaign",
            "cost": 2,
            "apply": lambda q: replace(q, eta=min(0.95, q.eta + 0.08)),
        },
        {
            "name": "rapid isolation",
            "cost": 2,
            "apply": lambda q: replace(q, delta=q.delta + 0.05),
        },
        {
            "name": "public alerts",
            "cost": 1,
            "apply": lambda q: replace(q, kappa=q.kappa + 15.0),
        },
        {
            "name": "mask distribution",
            "cost": 1,
            "apply": lambda q: replace(q, beta0=0.95 * q.beta0),
        },
        {
            "name": "isolation support",
            "cost": 1,
            "apply": lambda q: replace(q, rho=0.5 * q.rho),
        },
    ]


def apply_policy_set(base_params, cards):
    q = base_params
    names = []
    cost = 0
    for card in cards:
        q = card["apply"](q)
        names.append(card["name"])
        cost += card["cost"]
    return q, names, cost


cards = policy_cards()
rows = []
for r in range(len(cards) + 1):
    for chosen in combinations(cards, r):
        q, names, cost = apply_policy_set(p, chosen)
        if cost <= 5:
            t, Y = simulate(q, method="RK4", dt=0.25, T=180.0)
            s = summarize(t, Y, q)
            rows.append({
                "policy": " + ".join(names) if names else "no action",
                "cost": cost,
                "peak_hospital_load": s["peak_hospital_load"],
                "overload_days": s["overload_days"],
                "final_cumulative_cases": s["final_cumulative_cases"],
                "peak_I": s["peak_I"],
            })

policy_table = pd.DataFrame(rows).sort_values(
    ["overload_days", "peak_hospital_load", "final_cumulative_cases"]
).reset_index(drop=True)



policy_table = pd.DataFrame(rows).sort_values(
    ["overload_days", "peak_hospital_load", "final_cumulative_cases"]
).reset_index(drop=True)


display(HTML(
    policy_table.to_html(
        index=True,
        float_format=lambda x: f"{x:.3f}"
    )
))


fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    policy_table["final_cumulative_cases"],
    policy_table["peak_hospital_load"],
)

ax.axhline(600, linestyle="--", label="hospital capacity")

ax.set_title("Policy tradeoff: all feasible policies")
ax.set_xlabel("Final cumulative cases")
ax.set_ylabel("Peak hospital load")
ax.legend()
plt.show()

# Mission 8 — The hidden outbreak parameters

The council receives noisy weekly reports of infectious cases. Two parameters are uncertain:

$
\beta_0 \quad \text{and} \quad \kappa.
$

Here $\beta_0$ controls baseline transmission, while $\kappa$ controls behavioral feedback.

The task is to search over possible values and find the pair that best explains the data.


### Questions

1. Where is the minimum of the surface?
2. What would a broad valley mean for policy decisions?
3. Can two different parameter pairs produce similar epidemic curves?

In [ ]:
def make_weekly_observations(seed=7):
    rng = np.random.default_rng(seed)
    true_params = replace(p, beta0=0.78, kappa=18.0)
    t_true, Y_true = simulate(true_params, method="RK4", dt=0.1, T=180.0)
    weeks = np.arange(7.0, 181.0, 7.0)
    true_I = interp_component(t_true, Y_true, weeks, "I")
    noise = rng.normal(0.0, 250.0, size=len(weeks))
    observed_I = np.maximum(true_I + noise, 0.0)
    return weeks, observed_I, true_params


weeks, observed_I, true_params = make_weekly_observations()
observations = pd.DataFrame({"day": weeks, "observed_I": observed_I})
display(observations.head())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(weeks, observed_I, label="weekly reports")
ax.set_title("Noisy weekly reports")
ax.set_xlabel("Day")
ax.set_ylabel("Reported infectious cases")
ax.legend()
plt.show()

In [ ]:
def calibration_loss(beta0, kappa, weeks, observed_I, base_params):
    q = replace(base_params, beta0=float(beta0), kappa=float(kappa))
    t, Y = simulate(q, method="RK4", dt=0.5, T=180.0)
    pred_I = interp_component(t, Y, weeks, "I")
    return float(np.mean((pred_I - observed_I) ** 2))


beta_grid = np.linspace(0.60, 0.92, 25)
kappa_grid = np.linspace(0.0, 50.0, 25)
loss = np.zeros((len(kappa_grid), len(beta_grid)))

for i, kappa in enumerate(kappa_grid):
    for j, beta0 in enumerate(beta_grid):
        loss[i, j] = calibration_loss(beta0, kappa, weeks, observed_I, p)

best_index = np.unravel_index(np.argmin(loss), loss.shape)
best_kappa = kappa_grid[best_index[0]]
best_beta0 = beta_grid[best_index[1]]
best_loss = loss[best_index]

print(f"Best beta0  = {best_beta0:.4f}")
print(f"Best kappa  = {best_kappa:.4f}")
print(f"Best loss   = {best_loss:.2f}")
print(f"Hidden beta0 = {true_params.beta0:.4f}")
print(f"Hidden kappa = {true_params.kappa:.4f}")

In [ ]:
B, K = np.meshgrid(beta_grid, kappa_grid)

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(B, K, loss, linewidth=0, antialiased=True, alpha=0.9)
ax.scatter([best_beta0], [best_kappa], [best_loss], s=80, label="best fit")
ax.set_title("Calibration loss surface")
ax.set_xlabel("beta0")
ax.set_ylabel("kappa")
ax.set_zlabel("mean squared error")
ax.view_init(elev=25, azim=-160)
ax.legend()
plt.show()

In [ ]:
p_fit = replace(p, beta0=best_beta0, kappa=best_kappa)
t_fit, Y_fit = simulate(p_fit, method="RK4", dt=0.1, T=180.0)
fit_I_weekly = interp_component(t_fit, Y_fit, weeks, "I")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(weeks, observed_I, label="weekly reports")
ax.plot(weeks, fit_I_weekly, marker="o", label="best fitted model")
ax.set_title("Observed data versus fitted curve")
ax.set_xlabel("Day")
ax.set_ylabel("Infectious cases")
ax.legend()
plt.show()

# Mission 9 — Harder mission: robust policy under uncertainty

The fitted parameters are not exact. The city must choose a policy that works not only for the best-fit scenario, but also for nearby scenarios.

We test each policy under a small uncertainty box around the fitted parameters:

$
\beta_0 \in \{0.95\widehat\beta_0,\widehat\beta_0,1.05\widehat\beta_0\},
$

$
\kappa \in \{\widehat\kappa-10,\widehat\kappa,\widehat\kappa+10\}.
$

### Questions

1. Which policy has the smallest worst-case overload?
2. Is it the same as the best policy from Mission 7?
3. Why might a robust policy be preferable to a policy that only works for one fitted model?

In [ ]:
scenario_params = []
for beta_factor in [0.95, 1.0, 1.05]:
    for kappa_shift in [-10.0, 0.0, 10.0]:
        scenario_params.append(
            replace(
                p,
                beta0=float(best_beta0 * beta_factor),
                kappa=float(max(0.0, best_kappa + kappa_shift)),
            )
        )

rows = []
for r in range(len(cards) + 1):
    for chosen in combinations(cards, r):
        q_base, names, cost = apply_policy_set(p, chosen)
        if cost <= 5:
            overloads = []
            peak_hospitals = []
            cumulative_cases = []
            for scenario in scenario_params:
                q_scenario, _, _ = apply_policy_set(scenario, chosen)
                t, Y = simulate(q_scenario, method="RK4", dt=0.5, T=180.0)
                s = summarize(t, Y, q_scenario)
                overloads.append(s["overload_days"])
                peak_hospitals.append(s["peak_hospital_load"])
                cumulative_cases.append(s["final_cumulative_cases"])
            rows.append({
                "policy": " + ".join(names) if names else "no action",
                "cost": cost,
                "worst_overload_days": max(overloads),
                "worst_peak_hospital_load": max(peak_hospitals),
                "worst_cumulative_cases": max(cumulative_cases),
            })

robust_table = pd.DataFrame(rows).sort_values(
    ["worst_overload_days", "worst_peak_hospital_load", "worst_cumulative_cases"]
).reset_index(drop=True)


# Sort all robust policy results
robust_table = robust_table.sort_values(
    ["worst_overload_days", "worst_peak_hospital_load", "worst_cumulative_cases"]
).reset_index(drop=True)

display(HTML(
    robust_table.to_html(
        index=True,
        float_format=lambda x: f"{x:.3f}"
    )
))

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    robust_table["worst_cumulative_cases"],
    robust_table["worst_peak_hospital_load"],
)

ax.axhline(600, linestyle="--", label="hospital capacity")

ax.set_title("Robust policy tradeoff: all feasible policies")
ax.set_xlabel("Worst-case cumulative cases")
ax.set_ylabel("Worst-case peak hospital load")
ax.legend()
plt.show()